# 🚀 LifeMate 2026: Chạy Planner & Safe Hash Guard trên Google Colab / VS Code

Notebook này hỗ trợ cả 2 chế độ:
1. **Chạy trực tiếp (Interactive Foreground) với Auto-Resume:** Nhìn thấy từng mẫu chạy trên màn hình. Nếu mất mạng hoặc laptop ngủ, chạy lại sẽ tự động tiếp tục (resume) từ mẫu dở dang mà không mất công chạy lại từ đầu.
2. **Chạy ngầm (Background Runner - GẬP MÁY ĐI NGỦ ĐƯỢC):** Chạy độc lập trên GPU cloud của Colab bằng `nohup`. Bạn có thể gập máy tính, tắt VS Code, GPU trên mây vẫn chạy liên tục cho đến khi xong đủ 300 mẫu.

In [8]:
# 1. Kiểm tra GPU và VRAM khả dụng
!nvidia-smi
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {device_name}')
    print(f'Total VRAM: {vram_gb:.2f} GB')

Sat Nov  7 10:31:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             16W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q -U vllm transformers peft accelerate datasets chunkformer

In [10]:
# 3. Clone hoặc cập nhật repository ZAIC_2026 trên nhánh planner/vuong
import os
from pathlib import Path

if not os.path.exists('ZAIC_2026'):
    !git clone https://github.com/Bamboohoccode/ZAIC_2026.git
    %cd ZAIC_2026
else:
    %cd /content/ZAIC_2026
    !git fetch origin

!git checkout planner/vuong
!git pull origin planner/vuong
print('\nCommit mới nhất trên nhánh planner/vuong:')
!git log -n 1 --oneline

Cloning into 'ZAIC_2026'...
remote: Enumerating objects: 425, done.
remote: Counting objects: 100% (425/425), done.
remote: Compressing objects: 100% (241/241), done.
remote: Total 425 (delta 179), reused 396 (delta 151), pack-reused 0 (from 0)
Receiving objects: 100% (425/425), 1.09 MiB | 5.63 MiB/s, done.
Resolving deltas: 100% (179/179), done.
/content/ZAIC_2026/ZAIC_2026
branch 'planner/vuong' set up to track 'origin/planner/vuong'.
Switched to a new branch 'planner/vuong'
From https://github.com/Bamboohoccode/ZAIC_2026
 * branch            planner/vuong -> FETCH_HEAD
Already up to date.

Commit mới nhất trên nhánh planner/vuong:
e0847f6 (HEAD -> planner/vuong, origin/planner/vuong) [pipeline & colab] ho tro co che checkpoint --resume va background runner nohup tranh mat session khi ngu may


In [11]:
# 4. Thiết lập đường dẫn dữ liệu data/ (Nhanh & Trực tiếp trong 1 giây)
import os, glob
from pathlib import Path

os.makedirs('data', exist_ok=True)

# 1. Nếu có file zip data.zip ở /content hoặc MyDrive -> giải nén tức thì
zip_paths = ['/content/data.zip', 'data.zip', '/content/drive/MyDrive/data.zip']
for z in zip_paths:
    if os.path.exists(z):
        print(f'📦 Đang giải nén {z}...')
        !unzip -q -o "{z}" -d .
        break

# 2. Kiểm tra các đường dẫn phổ biến
direct_candidates = [
    Path('data/train'),
    Path('/content/data/train'),
    Path('/content/train'),
    Path('/content/drive/MyDrive/data/train'),
    Path('/content/drive/MyDrive/ZAIC_2026/data/train'),
    Path('/content/drive/MyDrive/ZAIC_2026/train'),
    Path('/content/drive/MyDrive/train'),
    Path('../data/train'),
]

found_train = None
for p in direct_candidates:
    if (p / 'train.jsonl').exists():
        found_train = p.resolve()
        break

# 3. Quét nhanh tối đa 2 cấp thư mục nếu chưa thấy
if not found_train:
    for base in ['/content', '/content/drive/MyDrive']:
        if os.path.exists(base):
            for cand in glob.glob(f'{base}/*/train.jsonl') + glob.glob(f'{base}/*/*/train.jsonl'):
                found_train = Path(cand).parent.resolve()
                break
        if found_train:
            break

if found_train and str(found_train) != str(Path('data/train').resolve()):
    print(f'✅ Đã tìm thấy dữ liệu tại: {found_train}')
    if not os.path.exists('data/train'):
        os.symlink(str(found_train), 'data/train')
    pub_cand = found_train.parent / 'public_test'
    if (pub_cand / 'public_test.jsonl').exists() and not os.path.exists('data/public_test'):
        os.symlink(str(pub_cand), 'data/public_test')

if os.path.exists('data/train/train.jsonl'):
    print('✅ Thư mục data/train đã sẵn sàng:')
    !ls -l data/train | head -n 5
else:
    print('⚠️ Chưa tìm thấy train.jsonl! Hãy chạy: !ln -s "/duong/dan/train" data/train')

📦 Đang giải nén /content/drive/MyDrive/data.zip...
✅ Thư mục data/train đã sẵn sàng:
total 7216
drwxrwxrwx 1547 root root   40960 Nov  5 15:55 audio
drwxrwxrwx 1547 root root   40960 Nov  5 15:56 train_environment
-rw-rw-rw-    1 root root 7198841 Nov  5 15:54 train.jsonl
-rw-rw-rw-    1 root root   91423 Nov  5 15:54 train_tools_catalog.json


### 🎯 LỰA CHỌN CÁCH CHẠY PIPELINE:
- **Cách 5A (Khuyên dùng khi đi ngủ):** Chạy ngầm trong background với `nohup`. Bạn có thể **gập máy tính lại**, GPU Colab vẫn chạy tiếp đến khi xong!
- **Cách 5B (Khi đang ngồi máy):** Chạy trực tiếp trên màn hình có kèm cờ `--resume` (nếu lỡ tắt máy, chạy lại sẽ tự làm tiếp từ mẫu vừa dừng).

In [12]:
# 5A. [KHUYÊN DÙNG] CHẠY NGẦM BACKGROUND (GẬP MÁY / SLEEP LAPTOP THOẢI MÁI)
# Dọn dẹp tiến trình cũ để giải phóng 100% VRAM GPU
!pkill -9 -f 'lifemate.pipeline' || true
import os, torch, time
time.sleep(2)
os.makedirs('runs', exist_ok=True)

vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 16
gpu_mem = 0.88 if vram < 20 else 0.90
print(f'GPU VRAM: {vram:.1f} GB -> Cấu hình --gpu-mem {gpu_mem}')

# Chạy ngầm bằng nohup: không bị ngắt khi gập máy tính
!nohup python -m lifemate.pipeline \
    --root data \
    --split train \
    --text \
    --llm Qwen/Qwen3-4B-Instruct-2507 \
    --mapper none \
    --gpu-mem {gpu_mem} \
    --max-ctx 8192 \
    --verify-rounds 0 \
    --prompt prompts/gepa/vuong/planner_v4.txt \
    --ids eval/splits/dev_ids.txt \
    --resume \
    --out runs/dev_v4_guarded.jsonl > runs/dev_v4_guarded.nohup.log 2>&1 &

print('🚀 Tiến trình đã được đưa vào chạy ngầm trên GPU Cloud!')
print('Bạn có thể gập máy tính lại hoặc theo dõi tiến độ ở Cell 5C bên dưới.')

GPU VRAM: 22.0 GB -> Cấu hình --gpu-mem 0.9
🚀 Tiến trình đã được đưa vào chạy ngầm trên GPU Cloud!
Bạn có thể gập máy tính lại hoặc theo dõi tiến độ ở Cell 5C bên dưới.


In [13]:
# 5B. (Tuỳ chọn) CHẠY TRỰC TIẾP TRÊN MÀN HÌNH (CÓ TỰ ĐỘNG RESUME TIẾP TỤC)
# import os, torch
# os.makedirs('runs', exist_ok=True)
# vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 16
# gpu_mem = 0.88 if vram < 20 else 0.90
#
# !python -m lifemate.pipeline \
#     --root data \
#     --split train \
#     --text \
#     --llm Qwen/Qwen3-4B-Instruct-2507 \
#     --mapper none \
#     --gpu-mem {gpu_mem} \
#     --max-ctx 8192 \
#     --verify-rounds 0 \
#     --prompt prompts/gepa/vuong/planner_v4.txt \
#     --ids eval/splits/dev_ids.txt \
#     --resume \
#     --out runs/dev_v4_guarded.jsonl

In [14]:
# 5C. THEO DÕI TIẾN ĐỘ THỜI GIAN THỰC (Chạy cell này bất cứ lúc nào để kiểm tra)
import time, os
from IPython.display import clear_output

out_path = 'runs/dev_v4_guarded.jsonl'
log_path = 'runs/dev_v4_guarded.jsonl.log'

for _ in range(60):  # cập nhật mỗi 5s trong 5 phút
    clear_output(wait=True)
    done_count = 0
    if os.path.exists(out_path):
        with open(out_path, 'r', encoding='utf-8') as f:
            done_count = sum(1 for line in f if line.strip())
    
    print(f'📊 TIẾN ĐỘ: Đã xong {done_count}/300 mẫu dev ({done_count/300*100:.1f}%)')
    print('=' * 60)
    if os.path.exists(log_path):
        !tail -n 12 runs/dev_v4_guarded.jsonl.log
    
    is_running = os.popen("pgrep -f 'lifemate.pipeline'").read().strip()
    if is_running:
        print(f'\n⚡ Tiến trình GPU đang chạy (PID: {is_running})...')
    else:
        print('\n⏹️ Tiến trình đã hoàn thành xong toàn bộ hoặc đã dừng.')
        break
    time.sleep(5)

📊 TIẾN ĐỘ: Đã xong 7/300 mẫu dev (2.3%)
=== Lifemate Pipeline Run ===
Start: 2026-11-07T10:32:35.178717
Output: runs/dev_v4_guarded.jsonl | Trace: runs/dev_v4_guarded.jsonl.trace.jsonl | Log: runs/dev_v4_guarded.jsonl.log
Split: train | Total samples: 300
Prompt: prompts/gepa/vuong/planner_v4.txt
[1/300] train_0006 calls=1 14.4s
[2/300] train_0017 calls=2 2.6s
[3/300] train_0028 calls=2 20.9s
[4/300] train_0029 calls=1 12.0s
[5/300] train_0036 calls=0 1.5s
[6/300] train_0038 calls=0 2.1s
[7/300] train_0050 calls=1 3.4s

⚡ Tiến trình GPU đang chạy (PID: 15080
16683)...


In [15]:
# 6. Chấm điểm và phân tích lỗi chi tiết với eval/scorer.py
!python eval/scorer.py --sub runs/dev_v4_guarded.jsonl


===== KẾT QUẢ ĐÁNH GIÁ (N=7) =====
Mean Score: 0.7857
  act  : 0.7000 (n=5)
  empty: 1.0000 (n=2)

Phân loại lỗi:
  WRONG_TOOL_OR_ARGS  : 2 (28.6%)


In [16]:
# 7. Tự động sao lưu file kết quả và run log vào Google Drive
import os, shutil
from pathlib import Path

if os.path.exists('/content/drive/MyDrive'):
    backup_dir = Path('/content/drive/MyDrive/ZAIC_2026_runs')
    backup_dir.mkdir(parents=True, exist_ok=True)
    for fname in ['dev_v4_guarded.jsonl', 'dev_v4_guarded.jsonl.log', 'dev_v4_guarded.jsonl.trace.jsonl']:
        fpath = Path('runs') / fname
        if fpath.exists():
            shutil.copy(fpath, backup_dir / fname)
            print(f'✅ Đã lưu: {fname} -> {backup_dir}')
    print(f'Toàn bộ kết quả đã được sao lưu an toàn tại Google Drive: {backup_dir}')
else:
    print('File kết quả hiện nằm trực tiếp tại: runs/dev_v4_guarded.jsonl')

✅ Đã lưu: dev_v4_guarded.jsonl -> /content/drive/MyDrive/ZAIC_2026_runs
✅ Đã lưu: dev_v4_guarded.jsonl.log -> /content/drive/MyDrive/ZAIC_2026_runs
✅ Đã lưu: dev_v4_guarded.jsonl.trace.jsonl -> /content/drive/MyDrive/ZAIC_2026_runs
Toàn bộ kết quả đã được sao lưu an toàn tại Google Drive: /content/drive/MyDrive/ZAIC_2026_runs
